# Interactive Hindi Word Scansion & Syllable Grouping

This notebook loads the trained sequence-labeling model (`trained_model/syllable_grouping_model.joblib`) and provides an **interactive loop** where you can enter any Hindi word to get:
1. **Syllabification** (Akshara segmentation)
2. **Weight Assignment** (Laghu = 1, Guru = 2 per classical chandas / saṃyoga rules)
3. **Intrinsic vs. Positional Heaviness**
4. **Context-Aware Sequence Labeling** (BIO boundary prediction using Logistic Regression)
5. **Word-Formation Grouping & 2-Mātrā Scansion Structure**

The loop runs indefinitely until you type `exit`, `quit`, or `q`.


In [6]:
# Install required dependencies
# Run this cell if executing on Google Colab or in a fresh environment
!pip install -q pandas numpy scikit-learn joblib


In [8]:
import os
import sys
import joblib
import numpy as np
import pandas as pd

MODEL_PATH = "trained_model/syllable_grouping_model.joblib"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at '{MODEL_PATH}'. "
        "Please ensure syllable_grouping_training.ipynb has been run, or trained_model/ is present."
    )

print(f"Loading model bundle from {MODEL_PATH}...")
bundle = joblib.load(MODEL_PATH)

vectorizer = bundle["vectorizer"]
model = bundle["model"]
config = bundle["config"]

SEQ_ID_COL = config.get("SEQ_ID_COL", "sequence_id")
SYLLABLE_COL = config.get("SYLLABLE_COL", "syllable")
WEIGHT_COL = config.get("WEIGHT_COL", "weight")
HEAVY_COL = config.get("HEAVY_COL", "heaviness")
POSITION_COL = config.get("POSITION_COL", "position")
LABEL_COL = config.get("LABEL_COL", "label")

print("Model successfully loaded!")
print("Classes:", model.classes_)
print("Feature count:", len(vectorizer.get_feature_names_out()))


Loading model bundle from trained_model/syllable_grouping_model.joblib...
Model successfully loaded!
Classes: ['B' 'I']
Feature count: 7772


In [9]:
# ============================================================================
# Prosody Rules & Akshara Segmentation
# ============================================================================

SHORT_MATRAS = {'\u093F', '\u0941', '\u0943', '\u0945', '\u0949'}  # ि ु ृ ॅ ॉ
LONG_MATRAS  = {'\u093E', '\u0940', '\u0942', '\u0947', '\u0948', '\u094B', '\u094C', '\u0944'}  # ा ी ू े ै ो ौ ॄ

SHORT_INDEP_VOWELS = {'\u0905', '\u0907', '\u0909', '\u090B'}  # अ इ उ ऋ
LONG_INDEP_VOWELS  = {'\u0906', '\u0908', '\u090A', '\u090F', '\u0910', '\u0913', '\u0914', '\u0960'}  # आ ई ऊ ए ऐ ओ औ ॠ

ANUSVARA = '\u0902'      # ं
CHANDRABINDU = '\u0901'  # ँ
VISARGA = '\u0903'       # ः
VIRAMA = '\u094D'        # ्
NUKTA = '\u093C'         # ़

VOWEL_SIGNS = SHORT_MATRAS | LONG_MATRAS
INDEP_VOWELS = SHORT_INDEP_VOWELS | LONG_INDEP_VOWELS

HA = '\u0939'  # ह
NASAL_LATERAL_ONSETS = {'\u092E', '\u0928', '\u0932'}  # म न ल


def is_consonant(ch):
    o = ord(ch)
    return (0x0915 <= o <= 0x0939) or (0x0958 <= o <= 0x095F)


def syllabify(word):
    """Segments a Hindi word into orthographic syllables (aksharas)."""
    i, n = 0, len(word)
    syllables = []
    while i < n:
        ch = word[i]
        if ch in INDEP_VOWELS:
            long_ = ch in LONG_INDEP_VOWELS
            i += 1
            nasal = False
            if i < n and word[i] in (ANUSVARA, CHANDRABINDU, VISARGA):
                nasal = True
                i += 1
            syllables.append({
                'text': ch, 'long': long_, 'nasal': nasal, 'lead_conjunct': 0,
                'has_own_vowel': True, 'final_cons': None, 'onset_first': None
            })
            continue

        if is_consonant(ch):
            lead_conjunct = 0
            start = i
            dangling = False
            final_cons = None
            onset_first = ch
            cur_base = ch
            while True:
                j = i + 1
                if j < n and word[j] == NUKTA:
                    j += 1
                if j < n and word[j] == VIRAMA:
                    i = j + 1
                    lead_conjunct += 1
                    if i < n and is_consonant(word[i]):
                        cur_base = word[i]
                        continue
                    else:
                        dangling = True
                        break
                else:
                    final_cons = cur_base
                    i = j
                    break

            long_ = False
            nasal = False
            has_own_vowel = not dangling
            if not dangling and i < n and word[i] in VOWEL_SIGNS:
                long_ = word[i] in LONG_MATRAS
                i += 1
            if not dangling and i < n and word[i] in (ANUSVARA, CHANDRABINDU, VISARGA):
                nasal = True
                i += 1

            syllables.append({
                'text': word[start:i], 'long': long_, 'nasal': nasal,
                'lead_conjunct': lead_conjunct, 'has_own_vowel': has_own_vowel,
                'final_cons': final_cons,
                'onset_first': onset_first if lead_conjunct >= 1 else None
            })
            continue

        i += 1  # Skip unknown chars/spaces/punctuation
    return syllables


def assign_weights_and_heaviness(syllables):
    """Assigns classical chandas weights (1/2) and categorizes heaviness."""
    n = len(syllables)
    weights = []
    heaviness = []
    for s in syllables:
        if not s['has_own_vowel']:
            weights.append(1)
            heaviness.append('light')
        elif s['long'] or s['nasal']:
            weights.append(2)
            heaviness.append('intrinsic')
        else:
            weights.append(1)
            heaviness.append('light')

    for idx in range(1, n):
        s = syllables[idx]
        is_mnl_ha_exception = (s['final_cons'] == HA and s['onset_first'] in NASAL_LATERAL_ONSETS)
        if s['lead_conjunct'] >= 1 and s['has_own_vowel'] and not is_mnl_ha_exception:
            weights[idx - 1] = 2
            heaviness[idx - 1] = 'positional'

    return weights, heaviness


In [10]:
# ============================================================================
# Feature Extraction & Prediction Pipeline
# ============================================================================

def safe_str(x):
    if pd.isna(x):
        return "<UNK>"
    return str(x)


def make_features(rows, i):
    r = rows[i]
    features = {
        "bias": 1,
        "syllable": safe_str(r[SYLLABLE_COL]),
        "weight": safe_str(r[WEIGHT_COL]),
        "heaviness": safe_str(r[HEAVY_COL]),
        "position": safe_str(r[POSITION_COL]),
        "position_numeric": float(r[POSITION_COL])
            if str(r[POSITION_COL]).replace(".", "", 1).isdigit()
            else 0.0,
        "is_first": int(i == 0),
        "is_last": int(i == len(rows) - 1),
        "prev_syllable": safe_str(rows[i-1][SYLLABLE_COL]) if i > 0 else "<START>",
        "prev_weight": safe_str(rows[i-1][WEIGHT_COL]) if i > 0 else "<START>",
        "prev_heaviness": safe_str(rows[i-1][HEAVY_COL]) if i > 0 else "<START>",
        "next_syllable": safe_str(rows[i+1][SYLLABLE_COL]) if i < len(rows)-1 else "<END>",
        "next_weight": safe_str(rows[i+1][WEIGHT_COL]) if i < len(rows)-1 else "<END>",
        "next_heaviness": safe_str(rows[i+1][HEAVY_COL]) if i < len(rows)-1 else "<END>",
    }

    for offset in [-2, -1, 1, 2]:
        j = i + offset
        if 0 <= j < len(rows):
            prefix = f"offset_{offset}_"
            features[prefix + "syllable"] = safe_str(rows[j][SYLLABLE_COL])
            features[prefix + "weight"] = safe_str(rows[j][WEIGHT_COL])
            features[prefix + "heaviness"] = safe_str(rows[j][HEAVY_COL])
        else:
            prefix = f"offset_{offset}_"
            features[prefix + "syllable"] = "<BOUNDARY>"
            features[prefix + "weight"] = "<BOUNDARY>"
            features[prefix + "heaviness"] = "<BOUNDARY>"

    return features


def predict_word(word, verbose=True):
    """Analyzes a single Hindi word and returns scansion grouping."""
    word = word.strip()
    syl = syllabify(word)
    if not syl:
        print(f"Error: Could not extract syllables from '{word}'.")
        return None

    weights, heaviness = assign_weights_and_heaviness(syl)
    rows = []
    for i, (s, w, h) in enumerate(zip(syl, weights, heaviness)):
        rows.append({
            SYLLABLE_COL: s['text'],
            WEIGHT_COL: w,
            HEAVY_COL: h,
            POSITION_COL: i
        })

    X_seq = [make_features(rows, i) for i in range(len(rows))]
    X_vec = vectorizer.transform(X_seq)
    preds = model.predict(X_vec)
    probs = model.predict_proba(X_vec)

    # Reconstruct syllable groups and scansion weights
    syllable_groups = []
    weight_groups = []
    curr_syl = []
    curr_w = []

    for s, w, lbl in zip(syl, weights, preds):
        if lbl == 'B' and curr_syl:
            syllable_groups.append(curr_syl)
            weight_groups.append(curr_w)
            curr_syl = []
            curr_w = []
        curr_syl.append(s['text'])
        curr_w.append(w)

    if curr_syl:
        syllable_groups.append(curr_syl)
        weight_groups.append(curr_w)

    group_chunks = [''.join(g) for g in syllable_groups]
    group_str = ' | '.join(group_chunks)
    scansion_str = ' | '.join(str(w) for w in weight_groups)

    if verbose:
        print("\n" + "=" * 65)
        print(f"WORD: {word}")
        print("=" * 65)
        
        detail_rows = []
        for i in range(len(rows)):
            p = dict(zip(model.classes_, probs[i]))
            detail_rows.append({
                "Pos": i,
                "Syllable": rows[i][SYLLABLE_COL],
                "Weight": f"{'Guru' if rows[i][WEIGHT_COL] == 2 else 'Laghu'} ({rows[i][WEIGHT_COL]})",
                "Heaviness": rows[i][HEAVY_COL],
                "BIO Tag": preds[i],
                "P(B)": f"{p.get('B', 0):.4f}",
                "P(I)": f"{p.get('I', 0):.4f}"
            })
        
        display(pd.DataFrame(detail_rows))
        print(f"\n  ➤ Formed Word Groups : {group_str}")
        print(f"  ➤ 2-Mātrā Scansion    : {scansion_str}")
        print(f"  ➤ Flat Weight Sequence: {weights}")
        print("=" * 65)

    return {
        "word": word,
        "syllables": [s['text'] for s in syl],
        "weights": weights,
        "heaviness": heaviness,
        "preds": [str(p) for p in preds],
        "groups": group_chunks,
        "scansion": weight_groups
    }


In [11]:
# ============================================================================
# Interactive Loop (Runs continuously until you enter 'exit', 'quit', or 'q')
# ============================================================================

print("=" * 65)
print("HINDI SCANSION & GROUPING INTERACTIVE TOOL")
print("Type any Hindi word (e.g. कंकरीला, कक्षा, बिजली, निग्रह, दिवाली, बात).")
print("To terminate the session, type 'exit', 'quit', or 'q'.")
print("=" * 65)

while True:
    try:
        user_input = input("\nEnter Hindi word: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nSession terminated.")
        break

    if not user_input or user_input.lower() in {"exit", "quit", "q", "stop", "end"}:
        print("Session terminated. Goodbye!")
        break

    predict_word(user_input, verbose=True)


HINDI SCANSION & GROUPING INTERACTIVE TOOL
Type any Hindi word (e.g. कंकरीला, कक्षा, बिजली, निग्रह, दिवाली, बात).
To terminate the session, type 'exit', 'quit', or 'q'.

WORD: कंकड़


,Pos,Syllable,Weight,Heaviness,BIO Tag,P(B),P(I)
0,0,कं,Guru (2),intrinsic,B,1.0000,0.0000
1,1,क,Laghu (1),light,B,0.8940,0.1060
2,2,ड़,Laghu (1),light,I,0.0036,0.9964



  ➤ Formed Word Groups : कं | कड़
  ➤ 2-Mātrā Scansion    : [2] | [1, 1]
  ➤ Flat Weight Sequence: [2, 1, 1]
Error: Could not extract syllables from 'suraj'.

WORD: फैंसी


,Pos,Syllable,Weight,Heaviness,BIO Tag,P(B),P(I)
0,0,फैं,Guru (2),intrinsic,B,1.0000,0.0000
1,1,सी,Guru (2),intrinsic,B,0.9992,0.0008



  ➤ Formed Word Groups : फैं | सी
  ➤ 2-Mātrā Scansion    : [2] | [2]
  ➤ Flat Weight Sequence: [2, 2]
Session terminated. Goodbye!
